In [1]:
# Task 1 (Section B) — Step 1: load the Twi sentences from Google Drive

import os, shutil

# If Drive isn't connected, clear anything left in its spot so it can connect
if not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

# Connect Google Drive (a pop-up will ask for permission)
from google.colab import drive
drive.mount('/content/drive')

# The folder where your Twi file lives
folder = "/content/drive/MyDrive/Ashesi/ICS554_Prosit1"

# Open the file and read every line into a list called all_text
# Each line is one Twi sentence; strip() removes the invisible line-break at the end
with open(folder + "/twi_sentences.txt", encoding="utf-8") as f:
    all_text = [line.strip() for line in f if line.strip()]

# Show how many sentences we have, and the first three
print("Sentences loaded:", len(all_text))
for sentence in all_text[:3]:
    print(sentence)

Mounted at /content/drive
Sentences loaded: 79566
Ɛyɛ me nwanwa sɛ yɛbɛtumi aka saa nsɛm yi.
Mma ɔkraman no ntu nkɔto kwaeɛ no so efisɛ ɔbɛhwerɛ ne nkwa nso.
Yɛnhwɛ mma yɛn future no nni nnyinasoɔ a ɛyɛ den na mmom yɛn nna adwuma a ɛbɛma yɛn society no ayɛ baako.


In [2]:
# Task 1 (Section B) — Step 2: look at the data

# Split every sentence into words and collect them
words = [w for s in all_text for w in s.split()]

print("Sentences:        ", len(all_text))
print("Total words:      ", len(words))
print("Different words:  ", len(set(words)))
print("Duplicate sentences:", len(all_text) - len(set(all_text)))

Sentences:         79566
Total words:       1129128
Different words:   41492
Duplicate sentences: 0


In [3]:
# Task 1 (Section B) — Step 3: clean the data
import string

def clean(sentence):
    sentence = sentence.lower()                  # Ɛyɛ -> ɛyɛ
    for p in string.punctuation:                 # remove . , ? ! etc.
        sentence = sentence.replace(p, "")
    return " ".join(sentence.split())            # tidy extra spaces

cleaned = [clean(s) for s in all_text]
cleaned = [s for s in cleaned if s]              # drop empty lines
cleaned = list(dict.fromkeys(cleaned))           # drop near-duplicates

words = [w for s in cleaned for w in s.split()]
print("Sentences after cleaning:", len(cleaned))
print("Different words:         ", len(set(words)))
print(cleaned[:2])

Sentences after cleaning: 79550
Different words:          31056
['ɛyɛ me nwanwa sɛ yɛbɛtumi aka saa nsɛm yi', 'mma ɔkraman no ntu nkɔto kwaeɛ no so efisɛ ɔbɛhwerɛ ne nkwa nso']


In [4]:
# Task 1 (Section B) — Step 4: split into train, dev and test
import random

random.seed(42)                    # same shuffle every run
shuffled = cleaned[:]
random.shuffle(shuffled)           # mix the order

n = len(shuffled)
train = shuffled[: int(n * 0.8)]
dev   = shuffled[int(n * 0.8) : int(n * 0.9)]
test  = shuffled[int(n * 0.9) :]

print("Train:", len(train))
print("Dev:  ", len(dev))
print("Test: ", len(test))

Train: 63640
Dev:   7955
Test:  7955


In [5]:
# Task 1 (Section B) — Step 5: count words and word pairs (training)
from collections import Counter

unigrams = Counter()     # single words
bigrams = Counter()      # pairs of words

for sentence in train:                       # training data only
    tokens = ["<s>"] + sentence.split() + ["</s>"]
    for i in range(len(tokens)):
        unigrams[tokens[i]] += 1
        if i > 0:
            bigrams[(tokens[i-1], tokens[i])] += 1

V = len(unigrams)                            # vocabulary size
print("Different words in training:", V)
print("Different word pairs:       ", len(bigrams))

Different words in training: 27659
Different word pairs:        196491


In [6]:
# Task 1 (Section B) — Step 6: predict the next word

def predict_next(word, top=5):
    # every pair that starts with this word, and its count
    followers = {w2: c for (w1, w2), c in bigrams.items() if w1 == word}
    total = unigrams[word]                      # how often the word appeared
    best = sorted(followers.items(), key=lambda x: x[1], reverse=True)[:top]
    for w2, c in best:
        print(f"{word} → {w2}   ({c}/{total} = {c/total:.1%})")

predict_next("me")

me → ne   (799/11228 = 7.1%)
me → werɛ   (564/11228 = 5.0%)
me → ya   (516/11228 = 4.6%)
me → ho   (477/11228 = 4.2%)
me → sɛ   (414/11228 = 3.7%)


In [7]:
predict_next("me")
predict_next("wo")
predict_next("sika")

me → ne   (799/11228 = 7.1%)
me → werɛ   (564/11228 = 5.0%)
me → ya   (516/11228 = 4.6%)
me → ho   (477/11228 = 4.2%)
me → sɛ   (414/11228 = 3.7%)
wo → ho   (592/4010 = 14.8%)
wo → nsa   (212/4010 = 5.3%)
wo → </s>   (155/4010 = 3.9%)
wo → ne   (119/4010 = 3.0%)
wo → nni   (91/4010 = 2.3%)
sika → a   (691/3210 = 21.5%)
sika → no   (555/3210 = 17.3%)
sika → ne   (213/3210 = 6.6%)
sika → </s>   (144/3210 = 4.5%)
sika → nko   (128/3210 = 4.0%)


In [8]:
predict_next("werɛ")

werɛ → how   (267/756 = 35.3%)
werɛ → fi   (104/756 = 13.8%)
werɛ → mfi   (62/756 = 8.2%)
werɛ → aho   (51/756 = 6.7%)
werɛ → remfi   (33/756 = 4.4%)


In [9]:
# Task 1 (Section B) — Step 7: fix the "never seen it" problem (smoothing)

def prob(prev, word):
    # plain count ÷ count — unseen pairs get 0
    return bigrams[(prev, word)] / unigrams[prev]

def prob_k(prev, word, k):
    # add k to every count, so nothing is ever 0
    return (bigrams[(prev, word)] + k) / (unigrams[prev] + k * V)

print("No smoothing:  ", prob("me", "nintendo"))
print("With k = 0.01: ", prob_k("me", "nintendo", 0.01))

No smoothing:   0.0
With k = 0.01:  8.692182859189246e-07


In [10]:
# Task 1 (Section B) — Step 8: score the model (perplexity) and choose k
import math

def perplexity(sentences, k):
    total_log, count = 0.0, 0
    for s in sentences:
        tokens = ["<s>"] + s.split() + ["</s>"]
        for i in range(1, len(tokens)):
            total_log += math.log(prob_k(tokens[i-1], tokens[i], k))  # add up surprise
            count += 1
    return math.exp(-total_log / count)          # turn average surprise into a score

for k in [0.0001, 0.001, 0.002, 0.005, 0.01, 0.1, 1.0]:
    print("k =", k, "→ dev perplexity:", round(perplexity(dev, k), 2))

k = 0.0001 → dev perplexity: 209.74
k = 0.001 → dev perplexity: 176.35
k = 0.002 → dev perplexity: 173.44
k = 0.005 → dev perplexity: 176.23
k = 0.01 → dev perplexity: 184.85
k = 0.1 → dev perplexity: 291.35
k = 1.0 → dev perplexity: 793.17


In [11]:
# Task 1 (Section B) — Step 9: compare with unigram and trigram

N = sum(unigrams.values())                       # total words in training

# Unigram: ignores the previous word
def unigram_perplexity(sentences, k):
    total_log, count = 0.0, 0
    for s in sentences:
        for w in s.split() + ["</s>"]:
            total_log += math.log((unigrams[w] + k) / (N + k * V))
            count += 1
    return math.exp(-total_log / count)

# Trigram: count triples, looking two words back (training data only)
trigrams, pair_context = Counter(), Counter()
for s in train:
    tokens = ["<s>", "<s>"] + s.split() + ["</s>"]
    for i in range(2, len(tokens)):
        trigrams[(tokens[i-2], tokens[i-1], tokens[i])] += 1
        pair_context[(tokens[i-2], tokens[i-1])] += 1

def trigram_perplexity(sentences, k):
    total_log, count = 0.0, 0
    for s in sentences:
        tokens = ["<s>", "<s>"] + s.split() + ["</s>"]
        for i in range(2, len(tokens)):
            p = (trigrams[(tokens[i-2], tokens[i-1], tokens[i])] + k) / \
                (pair_context[(tokens[i-2], tokens[i-1])] + k * V)
            total_log += math.log(p)
            count += 1
    return math.exp(-total_log / count)

# Try several k for each model, keep the best (dev only)
ks = [0.0001, 0.0005, 0.001, 0.002, 0.01, 0.1, 1.0]
best_uni = min((unigram_perplexity(dev, k), k) for k in ks)
best_tri = min((trigram_perplexity(dev, k), k) for k in ks)

print(f"Unigram: {best_uni[0]:.2f}  (k = {best_uni[1]})")
print(f"Bigram:  {perplexity(dev, 0.002):.2f}  (k = 0.002)")
print(f"Trigram: {best_tri[0]:.2f}  (k = {best_tri[1]})")

Unigram: 421.10  (k = 1.0)
Bigram:  173.44  (k = 0.002)
Trigram: 492.45  (k = 0.001)


In [12]:
# Task 1 (Section B) — Step 9b: try 4-grams and 5-grams

def build_ngrams(n):
    grams, context = Counter(), Counter()
    for s in train:                                         # training data only
        tokens = ["<s>"] * (n - 1) + s.split() + ["</s>"]
        for i in range(n - 1, len(tokens)):
            grams[tuple(tokens[i-n+1 : i+1])] += 1          # the n words together
            context[tuple(tokens[i-n+1 : i])] += 1          # the words before the last
    return grams, context

def ngram_perplexity(sentences, n, grams, context, k):
    total_log, count = 0.0, 0
    for s in sentences:
        tokens = ["<s>"] * (n - 1) + s.split() + ["</s>"]
        for i in range(n - 1, len(tokens)):
            p = (grams[tuple(tokens[i-n+1 : i+1])] + k) / \
                (context[tuple(tokens[i-n+1 : i])] + k * V)
            total_log += math.log(p)
            count += 1
    return math.exp(-total_log / count)

for n in [4, 5]:
    grams, context = build_ngrams(n)
    best = min((ngram_perplexity(dev, n, grams, context, k), k) for k in ks)
    print(f"{n}-gram: {best[0]:.2f}  (k = {best[1]})   different sequences: {len(grams)}")

4-gram: 1896.54  (k = 0.0005)   different sequences: 661696
5-gram: 4396.53  (k = 0.0005)   different sequences: 753372


In [13]:
# Task 1 (Section B) — Step 10: final exam on the test set (run once)

print("Dev perplexity: ", round(perplexity(dev, 0.002), 2))
print("Test perplexity:", round(perplexity(test, 0.002), 2))

Dev perplexity:  173.44
Test perplexity: 175.27
